# TIAGo SmolVLA training — ADR-0029
Local CPU compatibility is measured; Colab GPU throughput and end-to-end training remain unverified. This notebook refuses to train without the successful TRAIN dataset approval and its exact statistics hash. It resumes complete numeric checkpoints including optimizer, scheduler, RNG and step; no symlinks or inference tunnel.

Copy the prepared bundle folder (`source.zip`, `dataset-train-v1.zip`, `bundle-status.json`) to `MyDrive/vlarobot_smolvla/`. Google Drive authorization is the only expected interactive step. Run the cells in order. Never substitute diagnostic/failure recordings for the approved dataset.


In [ ]:
from pathlib import Path

from google.colab import drive

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/vlarobot_smolvla")
DATASET = Path("/content/vlarobot-data")  # recreated identically after reconnect
TRAIN_OUT = DRIVE / "training-tiago-v1"
SOURCE = Path("/content/VLA-Project")

## Install the isolated measured Python 3.11 stack
The runtime kernel may use another Python version. All project commands below run in the explicit Python 3.11 environment. GPU wheel/driver compatibility is checked before training. Installation does not start training.


In [ ]:
import hashlib
import json
import shutil
import subprocess
import sys
import zipfile

subprocess.run([sys.executable, "-m", "pip", "install", "uv==0.13.7"], check=True)
UV = [sys.executable, "-m", "uv"]
MANIFEST_SHA256 = "b335fa8a4825688fcfeb539dd7c9539c73dccc35bcf98892c0a94482d79939ff"
manifest_bytes = (DRIVE / "bundle-status.json").read_bytes()
if hashlib.sha256(manifest_bytes).hexdigest() != MANIFEST_SHA256:
    raise RuntimeError("Approved bundle-status.json fingerprint changed")
manifest = json.loads(manifest_bytes)
if manifest["status"] != "approved-payload-ready-awaiting-Colab-authentication":
    raise RuntimeError("Complete approved 80-success payload is not ready")
STAGING = Path("/content/vla-payloads")
STAGING.mkdir(exist_ok=True)
for name, key, destination in [
    ("source.zip", "source_zip_sha256", SOURCE),
    ("dataset-train-v1.zip", "dataset_zip_sha256", DATASET),
]:
    local = STAGING / name
    shutil.copyfile(DRIVE / name, local)
    with local.open("rb") as stream:
        if hashlib.file_digest(stream, "sha256").hexdigest() != manifest[key]:
            raise RuntimeError("Payload fingerprint mismatch: " + name)
    if destination.exists():
        shutil.rmtree(destination)  # scratch only; Drive checkpoints remain intact
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(local) as archive:
        if any(
            Path(info.filename).is_absolute() or ".." in Path(info.filename).parts
            for info in archive.infolist()
        ):
            raise RuntimeError("Unsafe archive member")
        archive.extractall(destination)
# Webots-free training still needs ffmpeg shared libraries for installed torchcodec.
if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "-qq", "update"], check=True)
    subprocess.run(["apt-get", "-qq", "install", "-y", "ffmpeg"], check=True)
PYTHON = Path("/content/vla-env/bin/python")
if not PYTHON.is_file():
    version = (SOURCE / ".python-version").read_text(encoding="utf-8").strip()
    subprocess.run([*UV, "venv", "--python", version, "/content/vla-env"], check=True)
subprocess.run(
    [
        *UV,
        "pip",
        "install",
        "--python",
        str(PYTHON),
        "-r",
        str(SOURCE / "model/compat/requirements-colab-py311.txt"),
    ],
    check=True,
)
subprocess.run(
    [*UV, "pip", "install", "--python", str(PYTHON), "--no-deps", "-e", str(SOURCE)], check=True
)
versions = subprocess.run(
    [*UV, "pip", "freeze", "--python", str(PYTHON)],
    check=True,
    capture_output=True,
    text=True,
    encoding="utf-8",
)
(DRIVE / "colab-installed.txt").write_text(versions.stdout, encoding="utf-8")
subprocess.run(
    [
        str(PYTHON),
        "-c",
        "import torch, torchcodec; assert torch.cuda.is_available(); "
        "assert 'T4' in torch.cuda.get_device_name(0), 'Select the authorized T4 runtime'; "
        "print(torch.__version__, torch.cuda.get_device_name(0))",
    ],
    check=True,
)

## Verify the actual dataset before training
The approval must come from the physically audited export and real-statistics installed forward check. Approximately 80 successful TRAIN episodes are required by the prepared workflow; no validation/test episodes or failed trajectories qualify.


In [ ]:
approval_check = """
from pathlib import Path
from vlarobot.vla.training import validate_training_approval
import sys
approval = validate_training_approval(Path(sys.argv[1]))
print('Verified successful TRAIN episodes:', len(approval['episodes']))
print('Camera:', approval['camera_key'], 'action:', approval['action_names'])
"""
subprocess.run([str(PYTHON), "-c", approval_check, str(DATASET)], check=True)

## T4 preflight — run before production training
This uses the installed SmolVLA training/update path at batch 8, seed 0 and the same
20,000-step scheduler horizon. It performs 10 diagnostic optimizer updates, writes a
complete checkpoint, restarts a fresh process, restores and verifies model/optimizer/
scheduler/RNG state, then performs 10 more updates. Only this isolated diagnostic uses
10-step checkpoint intervals; production remains **1,000**. Diagnostic weights are
labelled ineligible for production export/evaluation. Peak CUDA memory, throughput,
checkpoint cost and 20,000-step projections are saved to Drive. Inspect these results
before running the next training cell; a failed preflight stops here without tuning.


In [ ]:
# Embedded companion diagnostic: no fourth payload file and no change to approved archives.
PREFLIGHT_SCRIPT = Path("/content/t4_preflight.py")
PREFLIGHT_SCRIPT.write_text(
    (
        '"""Short isolated real SmolVLA training/resume diagnost'
        'ic; approved payload stays immutable."""\n\nfrom __future'
        "__ import annotations\n\nimport argparse\nimport hashlib\ni"
        "mport json\nimport math\nimport statistics\nimport subproc"
        "ess\nimport sys\nfrom pathlib import Path\nfrom time impor"
        "t perf_counter\nfrom typing import Any\n\n\ndef write_json("
        "path: Path, value: Any) -> None:\n    path.write_text(js"
        'on.dumps(value, indent=2, allow_nan=False) + "\\n", enco'
        'ding="utf-8")\n\n\ndef summarize(rows: list[dict[str, Any]'
        "], saves: list[dict[str, Any]]) -> dict[str, Any]:\n    "
        '"""Estimate unchanged campaign cost from completed upda'
        'tes, excluding warmup per process."""\n    if not rows o'
        'r any(not r["actual_optimizer_update"] for r in rows):\n'
        '        raise ValueError("throughput requires completed'
        ' optimizer updates")\n    steady = [r for r in rows if r'
        '["update_in_process"] > 2]\n    if not steady or not sav'
        'es:\n        raise ValueError("need post-warmup updates '
        'and checkpoint timing")\n    seconds = statistics.mean(r'
        '["update_s"] + r["dataload_s"] for r in steady)\n    che'
        'ckpoint_s = max(r["write_and_integrity_s"] for r in sav'
        'es)\n    return {\n        "observed_optimizer_updates": '
        'len(rows),\n        "steady_steps_per_s": 1 / seconds,\n '
        '       "estimated_20000_steps_hours": (20000 * seconds '
        '+ 20 * checkpoint_s) / 3600,\n        "estimated_1000_st'
        'ep_interval_minutes": (1000 * seconds + checkpoint_s) /'
        ' 60,\n        "max_checkpoint_write_and_integrity_s": ch'
        'eckpoint_s,\n        "estimated_20_checkpoint_bytes": 20'
        ' * max(r.get("checkpoint_bytes", 0) for r in saves),\n  '
        '      "peak_allocated_bytes": max(r["peak_allocated_byt'
        'es"] for r in rows),\n        "peak_reserved_bytes": max'
        '(r["peak_reserved_bytes"] for r in rows),\n        "esti'
        'mate_excludes_download_install_and_session_restart": Tr'
        'ue,\n        "short_sample_not_session_guarantee": True,'
        '\n        "diagnostic_checkpoint_cadence_only": True,\n  '
        "  }\n\n\ndef worker(dataset: Path, out: Path, updates: int"
        ', phase: int) -> None:\n    """Use installed LeRobot mai'
        'n/update/save/load, observing actual optimizer hooks.""'
        '"\n    import torch\n    import yaml\n    from lerobot.scr'
        "ipts import lerobot_train as installed\n\n    import vlar"
        "obot.vla.training as training\n    import vlarobot.vla.t"
        "raining_evidence as evidence\n\n    if not torch.cuda.is_"
        'available():\n        raise RuntimeError("T4 preflight r'
        'equires authorized CUDA; CPU fallback is forbidden")\n  '
        '  config = yaml.safe_load(Path("configs/training.yaml")'
        '.read_text(encoding="utf-8"))\n    if config != {"steps"'
        ': 20000, "batch_size": 8, "seed": 0, "save_freq": 1000}'
        ':\n        raise ValueError("preflight requires the acce'
        'pted frozen training configuration")\n    target = updat'
        "es * phase\n    trackers: list[Any] = []\n    rows: list["
        "dict[str, Any]] = []\n    saves: list[dict[str, Any]] = "
        "[]\n    original_evidence = evidence.OptimizerStepEviden"
        "ce\n    original_update = installed.update_policy\n    or"
        "iginal_save = installed.save_checkpoint\n    original_lo"
        "ad = installed.load_training_state\n    original_mark = "
        "training.mark_checkpoint_complete\n\n    class Diagnostic"
        "Evidence(original_evidence):\n        def __init__(self,"
        " optimizer: Any, path: Path, **kwargs: Any) -> None:\n  "
        '          kwargs["identity"] = {**kwargs["identity"], "'
        'run_kind": "smolvla-t4-preflight"}\n            kwargs["'
        'record_every"] = 1\n            super().__init__(optimiz'
        "er, path, **kwargs)\n            trackers.append(self)\n\n"
        "    def observe_update(*args: Any, **kwargs: Any) -> An"
        "y:\n        tracker = trackers[-1]\n        before = trac"
        "ker.calls\n        torch.cuda.synchronize()\n        if b"
        "efore == 0:\n            if phase == 2:\n                "
        "from safetensors import safe_open\n\n                chec"
        "kpoint = training.latest_complete_checkpoint(out)\n     "
        '           accelerator = kwargs["accelerator"]\n        '
        "        policy = accelerator.unwrap_model(args[1], keep"
        "_fp32_wrapper=True)\n                state = policy.stat"
        "e_dict()\n                with safe_open(\n              "
        '      checkpoint / "pretrained_model/model.safetensors"'
        ', framework="pt"\n                ) as f:\n              '
        "      for name in f.keys():\n                        if "
        "not torch.equal(state[name].detach().cpu(), f.get_tenso"
        "r(name)):\n                            raise ValueError("
        '"restored model tensor differs: " + name)\n             '
        '   write_json(out / "t4-model-restored.json", {"all_sav'
        'ed_model_tensors_equal": True})\n            torch.cuda.'
        "reset_peak_memory_stats()\n        started = perf_counte"
        "r()\n        result = original_update(*args, **kwargs)\n "
        "       torch.cuda.synchronize()\n        if tracker.call"
        's != before + 1:\n            raise RuntimeError("instal'
        "led update did not complete exactly one optimizer updat"
        'e")\n        metrics = result[0]\n        loss = float(me'
        "trics.loss.val)\n        if not math.isfinite(loss):\n   "
        '         raise ValueError("nonfinite training loss")\n  '
        '      row = {\n            "phase": phase,\n            "'
        'global_step": tracker.start_step + tracker.calls,\n     '
        '       "update_in_process": tracker.calls,\n            '
        '"actual_optimizer_update": True,\n            "loss": lo'
        'ss,\n            "update_s": perf_counter() - started,\n '
        '           "dataload_s": float(metrics.dataloading_s.va'
        'l),\n            "peak_allocated_bytes": torch.cuda.max_'
        'memory_allocated(),\n            "peak_reserved_bytes": '
        "torch.cuda.max_memory_reserved(),\n        }\n        row"
        's.append(row)\n        with (out / "t4-updates.jsonl").o'
        'pen("a", encoding="utf-8") as stream:\n            strea'
        'm.write(json.dumps(row, allow_nan=False) + "\\n")\n      '
        "  return result\n\n    def observe_save(*args: Any, **kwa"
        "rgs: Any) -> Any:\n        started = perf_counter()\n    "
        "    result = original_save(*args, **kwargs)\n        sav"
        'es.append({"step": kwargs["step"], "started": started})'
        "\n        return result\n\n    def observe_load(*args: Any"
        ", **kwargs: Any) -> Any:\n        started = perf_counter"
        "()\n        result = original_load(*args, **kwargs)\n    "
        "    if result[0] != updates:\n            raise ValueErr"
        'or("restored global step differs from first diagnostic '
        'checkpoint")\n        from lerobot.datasets.utils import'
        " flatten_dict\n        from lerobot.utils.random_utils i"
        "mport serialize_rng_state\n        from safetensors.torc"
        "h import load_file\n\n        checkpoint = Path(args[0])\n"
        '        saved_rng = load_file(checkpoint / "training_st'
        'ate/rng_state.safetensors")\n        actual_rng = flatte'
        "n_dict(serialize_rng_state())\n        if set(saved_rng)"
        " != set(actual_rng) or any(\n            not torch.equal"
        "(saved_rng[k], actual_rng[k].cpu()) for k in saved_rng\n"
        '        ):\n            raise ValueError("restored Pytho'
        'n/NumPy/Torch/CUDA RNG differs from checkpoint")\n      '
        "  optimizer, scheduler = result[1:]\n        if not opti"
        "mizer.state or scheduler is None:\n            raise Val"
        'ueError("optimizer or scheduler restoration missing")\n '
        '       saved_optimizer = load_file(checkpoint / "traini'
        'ng_state/optimizer_state.safetensors")\n        actual_o'
        'ptimizer = flatten_dict({"state": optimizer.state_dict('
        ')["state"]})\n        if set(saved_optimizer) != set(act'
        "ual_optimizer) or any(\n            not torch.equal(save"
        "d_optimizer[k], actual_optimizer[k].cpu()) for k in sav"
        'ed_optimizer\n        ):\n            raise ValueError("r'
        'estored optimizer tensors differ from checkpoint")\n    '
        "    saved_scheduler = json.loads(\n            (checkpoi"
        'nt / "training_state/scheduler_state.json").read_text(e'
        'ncoding="utf-8")\n        )\n        if json.loads(json.d'
        "umps(scheduler.state_dict())) != saved_scheduler:\n     "
        '       raise ValueError("restored scheduler differs fro'
        'm checkpoint")\n        write_json(\n            out / "t'
        '4-restore.json",\n            {\n                "restore'
        'd_global_step": result[0],\n                "installed_r'
        'estore_completed": True,\n                "rng_tensors_e'
        'qual": True,\n                "optimizer_tensors_equal":'
        ' True,\n                "scheduler_state_equal": True,\n '
        '               "optimizer_state_entries": len(optimizer'
        '.state),\n                "scheduler_state": scheduler.s'
        'tate_dict(),\n                "restore_wall_s": perf_cou'
        'nter() - started,\n                "checkpoint_sha_verif'
        'ied_before_launch": True,\n            },\n        )\n    '
        "    return result\n\n    class DiagnosticBoundary(Excepti"
        "on):\n        pass\n\n    def finish_checkpoint(path: Path"
        ") -> Path:\n        result = original_mark(path)\n       "
        ' elapsed = perf_counter() - saves[-1].pop("started")\n  '
        '      saves[-1]["write_and_integrity_s"] = elapsed\n    '
        '    saves[-1]["checkpoint_bytes"] = sum(\n            p.'
        'stat().st_size for p in path.rglob("*") if p.is_file()\n'
        '        )\n        write_json(out / f"t4-phase-{phase}-s'
        'aves.json", saves)\n        if int(path.name) >= target:'
        "\n            raise DiagnosticBoundary\n        return re"
        "sult\n\n    evidence.OptimizerStepEvidence = DiagnosticEv"
        "idence\n    installed.update_policy = observe_update\n   "
        " installed.save_checkpoint = observe_save\n    installed"
        ".load_training_state = observe_load\n    training.mark_c"
        "heckpoint_complete = finish_checkpoint\n    # Diagnostic"
        "-only save cadence; model, batch, seed, loss, scheduler"
        " and 20k horizon unchanged.\n    # Production configs an"
        "d output are never edited or resumed from this diagnost"
        "ic directory.\n    try:\n        training.launch_approved"
        '_training(dataset, out, **{**config, "save_freq": updat'
        "es})\n    except DiagnosticBoundary:\n        pass\n    ch"
        "eckpoint = training.latest_complete_checkpoint(out)\n   "
        " if checkpoint is None or int(checkpoint.name) != targe"
        't:\n        raise RuntimeError("diagnostic boundary has '
        'no complete hash-verified checkpoint")\n    write_json(\n'
        '        out / f"t4-phase-{phase}.json",\n        {\n     '
        '       "gpu": torch.cuda.get_device_name(),\n           '
        ' "total_memory_bytes": torch.cuda.get_device_properties'
        '(0).total_memory,\n            "phase": phase,\n         '
        '   "final_global_step": target,\n            "updates_in'
        '_process": len(rows),\n            "complete_checkpoint"'
        ': str(checkpoint),\n            "test_split_used": False'
        ',\n            "diagnostic_only_not_production_training"'
        ": True,\n        },\n    )\n\n\ndef main() -> None:\n    pars"
        "er = argparse.ArgumentParser(description=__doc__)\n    p"
        'arser.add_argument("--dataset", type=Path, required=Tru'
        'e)\n    parser.add_argument("--out", type=Path, required'
        '=True)\n    parser.add_argument("--updates", type=int, d'
        'efault=10)\n    parser.add_argument("--worker-phase", ty'
        'pe=int, choices=(1, 2))\n    parser.add_argument("--avai'
        'lable-session-hours", type=float)\n    args = parser.par'
        "se_args()\n    if args.updates < 4 or args.updates >= 10"
        '00:\n        raise ValueError("diagnostic length must be'
        ' between 4 and 999 updates per process")\n    if args.wo'
        "rker_phase:\n        worker(args.dataset.resolve(), args"
        ".out.resolve(), args.updates, args.worker_phase)\n      "
        "  return\n    if args.out.exists():\n        raise FileEx"
        'istsError("preserve prior preflight; select a new versi'
        'oned output")\n    # LeRobot owns creation of the traini'
        "ng output, so supervisor evidence lives beside it.\n    "
        "args.out.parent.mkdir(parents=True, exist_ok=True)\n    "
        'frozen = Path("configs/training.yaml")\n    before = has'
        "hlib.sha256(frozen.read_bytes()).hexdigest()\n    starte"
        "d = perf_counter()\n    for phase in (1, 2):\n        wit"
        'h args.out.with_name(args.out.name + f".phase-{phase}.l'
        'og").open(\n            "x", encoding="utf-8"\n        ) '
        "as log:\n            try:\n                subprocess.run"
        "(\n                    [\n                        sys.exe"
        "cutable,\n                        str(Path(__file__).res"
        'olve()),\n                        "--dataset",\n         '
        "               str(args.dataset.resolve()),\n           "
        '             "--out",\n                        str(args.'
        'out.resolve()),\n                        "--updates",\n  '
        "                      str(args.updates),\n              "
        '          "--worker-phase",\n                        str'
        "(phase),\n                    ],\n                    std"
        "out=log,\n                    stderr=subprocess.STDOUT,\n"
        "                    check=True,\n                )\n     "
        "       except subprocess.CalledProcessError:\n          "
        "      log.flush()\n                log_path = Path(log.n"
        'ame)\n                print(\n                    "\\n".jo'
        'in(log_path.read_text(encoding="utf-8").splitlines()[-6'
        "0:]), flush=True\n                )\n                rais"
        "e\n    rows = [json.loads(line) for line in (args.out / "
        '"t4-updates.jsonl").read_text().splitlines()]\n    saves'
        " = [\n        r\n        for phase in (1, 2)\n        for "
        'r in json.loads((args.out / f"t4-phase-{phase}-saves.js'
        'on").read_text())\n    ]\n    if [r["global_step"] for r '
        "in rows] != list(range(1, args.updates * 2 + 1)):\n     "
        '   raise ValueError("diagnostic optimizer updates are n'
        'ot complete and contiguous")\n    if not (args.out / "t4'
        '-model-restored.json").is_file():\n        raise ValueEr'
        'ror("restored model has not been compared with saved ch'
        'eckpoint")\n    report = summarize(rows, saves)\n    repo'
        'rt["restore_verified"] = json.loads((args.out / "t4-res'
        'tore.json").read_text())\n    report["preflight_complete'
        '"] = True\n    report.update(\n        preflight_wall_s=p'
        "erf_counter() - started,\n        production_training_st"
        "arted=False,\n        diagnostic_only=True,\n        prod"
        "uction_config_sha256=before,\n    )\n    if hashlib.sha25"
        "6(frozen.read_bytes()).hexdigest() != before:\n        r"
        'aise ValueError("frozen training configuration changed '
        'during preflight")\n    if args.available_session_hours '
        'is not None:\n        report["available_session_hours"] '
        '= args.available_session_hours\n        report["estimate'
        'd_full_run_fits_session"] = (\n            report["estim'
        'ated_20000_steps_hours"] < args.available_session_hours'
        '\n        )\n        report["estimated_checkpoint_interva'
        'l_fits_session"] = (\n            report["estimated_1000'
        '_step_interval_minutes"] / 60 < args.available_session_'
        'hours\n        )\n    write_json(args.out / "t4-preflight'
        '-report.json", report)\n    print(json.dumps(report, ind'
        'ent=2))\n\n\nif __name__ == "__main__":\n    main()\n'
    ),
    encoding="utf-8",
)
PREFLIGHT_SHA = hashlib.sha256(PREFLIGHT_SCRIPT.read_bytes()).hexdigest()
PREFLIGHT_ROOT = DRIVE / "preflight"
PREFLIGHT_ROOT.mkdir(exist_ok=True)
PREFLIGHT_REPORT = None
for candidate in sorted(PREFLIGHT_ROOT.glob("t4-v*/t4-preflight-report.json")):
    report = json.loads(candidate.read_text(encoding="utf-8"))
    if report.get("companion_sha256") == PREFLIGHT_SHA and report.get("preflight_complete"):
        PREFLIGHT_REPORT = report
        print("Reusing completed diagnostic:", candidate)
        break
if PREFLIGHT_REPORT is None:
    index = 1
    while (PREFLIGHT_ROOT / f"t4-v{index:03d}").exists() or (
        PREFLIGHT_ROOT / f"t4-v{index:03d}.phase-1.log"
    ).exists():
        index += 1
    diagnostic = PREFLIGHT_ROOT / f"t4-v{index:03d}"
    subprocess.run(
        [str(PYTHON), str(PREFLIGHT_SCRIPT), "--dataset", str(DATASET), "--out", str(diagnostic)],
        cwd=SOURCE,
        check=True,
    )
    report_path = diagnostic / "t4-preflight-report.json"
    PREFLIGHT_REPORT = json.loads(report_path.read_text(encoding="utf-8"))
    PREFLIGHT_REPORT["companion_sha256"] = PREFLIGHT_SHA
    report_path.write_text(json.dumps(PREFLIGHT_REPORT, indent=2) + "\n", encoding="utf-8")
print(json.dumps(PREFLIGHT_REPORT, indent=2))
print("Session availability is not guaranteed. Compare the measured run/interval estimates")
print("with your available session. No training settings have been changed.")

## Train or resume
Rerun this cell after a disconnect using the same Drive paths. The guard selects the highest complete numeric checkpoint and restores all training state. An incomplete newer checkpoint is ignored. If no complete checkpoint exists, the launcher preserves the interrupted output and selects a new numbered restart automatically. Active jobs on the same host are protected against duplicate launch.

The launcher strictly uses the pinned base snapshot and the measured 8-state / 7-action / camera1 schema. State and action use dataset MEAN_STD normalization; gripper actions are absolute per-finger target metres. Save frequency is 1000 steps. Measure warm-up steps/s in the log before estimating T4 runtime; no A100 estimate is assumed.


In [ ]:
if PREFLIGHT_REPORT is None or not PREFLIGHT_REPORT.get("preflight_complete"):
    raise RuntimeError("A complete T4 optimizer/checkpoint preflight is required")
train_code = """
from pathlib import Path
from vlarobot.vla.training import launch_approved_training
import sys, yaml, json
import torch
import vlarobot.vla.training_evidence as evidence
OriginalEvidence = evidence.OptimizerStepEvidence
class ConsoleEvidence(OriginalEvidence):
    def post_step(self, optimizer, args, kwargs):
        super().post_step(optimizer, args, kwargs)
        if self.calls == 1 or (self.start_step+self.calls) % self.record_every == 0:
            record=json.loads(self.path.with_name('training-progress.json').read_text())
            print('ACTUAL OPTIMIZER UPDATE:',json.dumps(record),flush=True)
evidence.OptimizerStepEvidence=ConsoleEvidence
configuration = yaml.safe_load(Path(sys.argv[3]).read_text(encoding='utf-8'))
launch_approved_training(Path(sys.argv[1]), Path(sys.argv[2]), **configuration)
"""
# This explicit cell is authorized only after complete dataset approval and CUDA checks.
# Actual starts are evidenced by optimizer-steps.jsonl, never by a launch message.
subprocess.run(
    [
        str(PYTHON),
        "-u",
        "-c",
        train_code,
        str(DATASET),
        str(TRAIN_OUT),
        str(SOURCE / "configs/training.yaml"),
    ],
    cwd=SOURCE,
    check=True,
)

## Inspect recorded production optimizer evidence
The training cell prints `ACTUAL OPTIMIZER UPDATE` after the first completed update.
This cell reports the persisted first event and current progress. A forward-pass loss
or loaded model is not optimizer evidence. The training cell normally blocks until
20,000 steps or an interruption; then inspect these files and export below.


In [ ]:
for directory in sorted(DRIVE.glob("training-tiago-v1*")):
    if not directory.is_dir():
        continue
    events = directory / "optimizer-steps.jsonl"
    if events.exists():
        with events.open(encoding="utf-8") as stream:
            first = json.loads(next(stream))
        assert first["actual_post_step_hook"] and first["optimizer_step"] > 0
        assert first["run_kind"] == "smolvla-finetune"
        print("Production directory:", directory)
        print("First actual optimizer event:", json.dumps(first, indent=2))
        print("Current progress:", (directory / "training-progress.json").read_text())

## Export for local inference
Export the latest complete checkpoint without creating symlinks. This does not establish robot task success; the local model still requires the frozen VLA-only 24-episode evaluation and literal gate `min(0.40, 0.60 × repaired_scripted_rate)`.


In [ ]:
export_code = """
from pathlib import Path
import sys
from vlarobot.vla.training import latest_complete_checkpoint
from vlarobot.vla.training_runs import select_training_output
from vlarobot.vla.checkpoint_export import export_complete_checkpoint
training = select_training_output(Path(sys.argv[1]))
checkpoint = latest_complete_checkpoint(training)
if checkpoint is None:
    raise RuntimeError('No complete checkpoint to export')
out = Path(sys.argv[2]) / (training.name + '-step-' + checkpoint.name)
export_complete_checkpoint(training, out)
print('Portable trained local-inference checkpoint:', out)
"""
subprocess.run([str(PYTHON), "-c", export_code, str(TRAIN_OUT), str(DRIVE / "exports")], check=True)